In [9]:
import requests
import torch
import re
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [10]:
ds = load_dataset("KushT/bbc_news_multiclass_train_val_test")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 334 entries, 0 to 333
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    334 non-null    object
 1   label   334 non-null    int64 
dtypes: int64(1), object(1)
memory usage: 5.3+ KB


In [11]:
test['label'] = test['label'].apply(lambda x: 'business' if x == 0 else 'entertainment' if x == 1 else 'politics' if x == 2 else 'sport' if x == 3 else 'tech')

labels = test['label'].unique()

test

,text,label
0,Dogged Federer claims Dubai crown World number...,sport
1,UK troops on Ivory Coast standby Downing Stree...,politics
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment
3,Classy Henman makes winning start Tim Henman o...,sport
4,Mixed reaction to Man Utd offer Shares in Manc...,business
...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport
330,Budget Aston takes on Porsche British car make...,business
331,Hi-tech posters guide commuters Interactive po...,tech
332,Hotspot users gain free net calls People using...,tech


In [12]:
labels

array(['sport', 'politics', 'entertainment', 'business', 'tech'],
      dtype=object)

In [13]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_5852\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


81944576

In [14]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [15]:
def classify(text, labels):
    url = "http://localhost:11434/api/chat"
    
    messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multiclass classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Category classification of news articles. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ]
    
    start_time = time.time()

    try:
        response = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": True,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 3100
            }
        }, timeout=30)
        response_time = time.time() - start_time
        vram_usage = get_gpu_memory_usage()
        ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)
        response = response.json()
        response_text = response['message'].get('thinking', '') if 'message' in response else ''
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return "error", {}, 0, 0, 0, 0, f"API Error: {e}"
    
    if not response.get('done', False):
        print(f"Ollama returned an incomplete response: {response.get('error')}")
        return 'error', {}, response_time, 0, 0, 0, response.get('error', 'Incomplete response')
    
    if 'message' in response and 'content' in response['message']:
        classification_text = response['message']['content'].lower()
        print("Response fields:", ', '.join(response.keys()))
        print(response)
        total_time = response['total_duration'] / 1_000_000_000
    else:
        messages.append({"role": "assistant", "content": response_text + '</think>'})
        start_time2 = time.time()
        response2 = requests.post(url, json={
            "model": "deepseek-r1:1.5b",
            "messages": messages,
            "think": False,
            "stream": False,
            "options": {
                "temperature": 0,
                "num_predict": 430
            }
        })
        response_time += time.time() - start_time2
        response2 = response2.json()
        classification_text = response2['message']['content'].lower() if 'message' in response2 and 'content' in response2['message'] else ''
        total_time = response['total_duration'] / 1_000_000_000 + response2['total_duration'] / 1_000_000_000
    
    label_counts = {label: len(re.findall(r'\b' + re.escape(label.lower()) + r'\b', classification_text)) for label in labels}
    
    if all(count == label_counts[labels[0]] for count in label_counts.values()):
        content = 'error'
    else:
        content = max(label_counts, key=label_counts.get)
    
    print(f"Text: {text}")
    print(f"Response: {content}")
    
    return content, label_counts, response_time, vram_usage, ram_usage_bytes, total_time, response_text

In [16]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'label_counts','response_time', 'vram_usage', 'ram_usage', 'total_time', 'response_text']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_5852\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


Response fields: model, created_at, message, done_reason, done, total_duration, load_duration, prompt_eval_count, prompt_eval_duration, eval_count, eval_duration
{'model': 'deepseek-r1:1.5b', 'created_at': '2025-06-14T20:08:25.8793962Z', 'message': {'role': 'assistant', 'content': 'sport', 'thinking': "Okay, so I need to classify this text into one of the given categories: sport, politics, entertainment, business, or tech. Let me read through the text carefully.\n\nThe text is about a tennis match between Roger Federer and Ivan Ljubicic. It talks about their performance in the first set, how they played, and some specific moments like when Federer was struggling with his racket and Ljubicic's poor showing. The text also mentions that Ljubicic had a chance to play against him twice in two weeks, which boosted his confidence.\n\nNow, looking at the categories: sport is about sports events, politics is about government or societal issues, entertainment is about activities like watching sh

In [17]:
test.to_csv('results/deepseekR1_ZS_multiclass2.csv', index=False)
test

,text,label,prediction,label_counts,response_time,vram_usage,ram_usage,total_time,response_text
0,Dogged Federer claims Dubai crown World number...,sport,sport,"{'sport': 1, 'politics': 0, 'entertainment': 0...",3.780587,2571,77.960938,1.736838,"Okay, so I need to classify this text into one..."
1,UK troops on Ivory Coast standby Downing Stree...,politics,politics,"{'sport': 0, 'politics': 1, 'entertainment': 0...",3.687602,2559,78.332031,1.635208,"Okay, so I need to classify this news text int..."
2,Keanu Reeves given Hollywood star Actor Keanu ...,entertainment,entertainment,"{'sport': 0, 'politics': 0, 'entertainment': 1...",3.965654,2556,77.769531,1.904926,"Okay, so I need to classify this news article ..."
3,Classy Henman makes winning start Tim Henman o...,sport,sport,"{'sport': 1, 'politics': 0, 'entertainment': 0...",3.885764,2556,78.046875,1.853885,"Okay, so I need to classify this news text int..."
4,Mixed reaction to Man Utd offer Shares in Manc...,business,politics,"{'sport': 0, 'politics': 1, 'entertainment': 0...",5.320069,2554,78.812500,3.282025,"Okay, so I need to classify this news text int..."
...,...,...,...,...,...,...,...,...,...
329,Middlesbrough 2-2 Charlton A late header by te...,sport,sport,"{'sport': 1, 'politics': 0, 'entertainment': 0...",3.782181,2564,78.777344,1.715667,"Okay, so I need to classify this news article ..."
330,Budget Aston takes on Porsche British car make...,business,business,"{'sport': 0, 'politics': 0, 'entertainment': 0...",4.887323,2564,78.292969,2.836024,"Okay, so I need to classify this text into one..."
331,Hi-tech posters guide commuters Interactive po...,tech,tech,"{'sport': 0, 'politics': 0, 'entertainment': 0...",6.050859,2557,78.777344,3.997250,"Okay, so I need to classify this text into one..."
332,Hotspot users gain free net calls People using...,tech,tech,"{'sport': 0, 'politics': 0, 'entertainment': 0...",3.985981,2564,78.785156,1.944997,"Okay, so I need to classify this news text int..."


In [18]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.763473
F1 score: 0.782088
Precision: 0.828460
Recall: 0.763473


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [19]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 4.790796869529221
Average VRAM usage: 2530.925149700599
Average RAM usage: 78.28548372005989
Average total time: 2.7463799428143716


In [20]:
# save results to txt
with open('results/deepseekR1_ZS_multiclass2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')
    f.write(f'Lines classified: {len(test)}\n')